**This script clusters GTs on an imagined candidate site. The universal and basic models DEL prediction models can be trialled and different feature sets (damage severity vs environmental based) can be assigned.**

In [ ]:
#Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import joblib
import matplotlib as mpl


**Data preprocessing**

In [ ]:
#-----------------------------------------------------------------------------------INPUTS-----------------------------------------------------------------------------------------------------
# USER INPUTS - can change these before running

#for manual damage binning:
bin_width_pct = 11 # percentage deviation between bin boundaries

k_fixed = 7 #only use this line if not using optimum k

m = 'poly'                   # model type: 'poly' or 'gam'

approach = 'UNIVERSAL'      # model approach: 'UNIVERSAL' or 'basic' 
                            # basic performs better when applied to a known GT of only one type, whereas Universal is has more general application 

GT = 10 #this only matters for 'basic' approach and should be changed to match the GT model on site   

cluster_by = 'damage' #options are: iref, damage, all_env

node_damage_features = 'towerBase'  #options are: MxMy, towerBase, or all. This determines which nodes will be sued as features to cluster. - only relevent if cluster_by=damage

#path to the site data and models
site_data_path = 'add_path_here'

figure_path = 'add_{figure_path}_here'

#set the min amount of turbines per cluster
min_cluster_size = 2

#--------------------------------------------------------------------------------LOADING DATA---------------------------------------------------------------------------------------------------

#NO CHANGE BELOW THIS POINT

#this is the path to the DEL prediction model
damage_model_path = f'{site_data_path}{approach}_Models/{m}_{approach}_model_'

#this is the candidate site for clustering
GT_df = pd.read_csv(f'{site_data_path}candidate_site5.csv') #wind speed and inflow angle are directional

#this was the GT data set the candidate site was made from
GT10_df = pd.read_csv(f'{site_data_path}GT10_DB_list.csv')

#environmental variables
env_inputs = [
    'windSpeed',
    'iRef',
    'shearExp',
    'density',
    'inFlowAngle'
]

#number of turbines in the candidate site
no_turbs = len(GT_df)

all_nodes = ['blade_root_Mx_4', 'blade_root_Mx_9', 'blade_root_Mx_10', 'blade_root_Mx_14','blade_root_My_4', 'blade_root_My_9', 'blade_root_My_10', 'blade_root_My_14', 'tower_base_My_4', 'tower_base_My_9']

#This is the full available node/load/wohler combinations for damage predictions
nodes = ['blade_root', 'tower_base']
loads = ['Mx', 'My']
wohlers = [4, 9, 10, 14]

#different feature set methods
if node_damage_features == 'all':
    node_features = ['blade_root_Mx_4', 'blade_root_Mx_9', 'blade_root_Mx_10', 'blade_root_Mx_14','blade_root_My_4', 'blade_root_My_9', 'blade_root_My_10', 'blade_root_My_14', 'tower_base_My_4', 'tower_base_My_9']
elif node_damage_features == 'MxMy':
    node_features = ['blade_root_Mx_14','blade_root_My_14', 'tower_base_My_4']
elif node_damage_features == 'towerBase':
    node_features = ['tower_base_My_4']

#the different clustering methods trialled
methods = ['kmeans', 'spectral', 'agglomerative', 'gaussian']


**Defining Functions**

In [ ]:
def predict_damage_GT(GT_df, site_data_path, node_features):
    #This is the damage prediction function which predicts damage using the environmental variables on site - to then use the predictions for clustering feature
    
    damage_model_path = f'{site_data_path}{approach}_Models/{m}_{approach}_model_'
    damage_rows = []

    #extracting the evironmental conditions from the site
    for i, (_, row) in enumerate(GT_df.iterrows()):
        turbine = f'T{str(i+1).zfill(3)}'
        windSpeed = row['windSpeed']
        iRef = row['iRef']
        shearExp = row['shearExp']
        inFlowAngle = row['inFlowAngle']
        density = row.get('density', 1.225)

        #the predictor variables
        X_test = pd.DataFrame([[windSpeed, iRef, shearExp, density, inFlowAngle]],
                               columns=['windSpeed', 'iRef', 'shearExp', 'density', 'inFlowAngle'])

        damage_row = {
            'turbine': turbine,
            'windSpeed': windSpeed,
            'iRef': iRef,
            'shearExp': shearExp,
            'inFlowAngle': inFlowAngle,
            'density': density
        }

        #looping through all node/load/wohler subsets to apply prediction model to each
        for node in nodes:
            for load in loads:
                for wohler in wohlers:
                    #only for the set node features
                    if f'{node}_{load}_{wohler}' not in node_features:
                        continue
                    subset_name = f'{node}_{load}_{wohler}'
                    try:
                        #depends on which mode was selected
                        if approach == 'UNIVERSAL':
                            model = joblib.load(f'{damage_model_path}{subset_name}_combined_16.pkl')
                        elif approach == 'basic':
                            model = joblib.load(f'{damage_model_path}GT{GT}_{subset_name}.pkl')
                        y_pred = model.predict(X_test)
                        damage_row[subset_name] = y_pred[0]
                    except FileNotFoundError:
                        print(f'Model not found for {subset_name}')
                        damage_row[subset_name] = np.nan

        damage_rows.append(damage_row)
    #returns a df containing the predicted damage and corresponding env conditions
    return pd.DataFrame(damage_rows)

def find_elbow_k_turbines(damage_GT, method, k_tolerance=0.15, fixed_k=None, plot_max_k=None):
    #this function finds the optimum k for clustering using the elbow method
    X_d = damage_GT
    no_turbs = len(damage_GT)

    #setting the desired cluster range
    if cluster_by == 'iref':
        max_n = 7
        max_k = 8
    else:
        max_n = 10
        max_k = 11

    min_n = 2

    # wider range purely for plotting context; detection still only uses min_n-max_n below
    plot_k_max = plot_max_k if plot_max_k is not None else max_k
    k_range = range(2, plot_k_max)
    stds = []

    #trying all of the cluster algorithms specified
    for k in k_range:
        if method == 'kmeans':
            labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_d)
        elif method == 'spectral':
            labels = SpectralClustering(n_clusters=k, random_state=42).fit_predict(X_d)
        elif method == 'agglomerative':
            labels = AgglomerativeClustering(n_clusters=k).fit_predict(X_d)
        elif method == 'gaussian':
            labels = GaussianMixture(n_components=k, random_state=42).fit_predict(X_d)

        #the standard deviation is used vs k in the elbow plot
        cluster_stds = []
        for label in np.unique(labels):
            cluster_data = X_d[labels == label]
            cluster_stds.append(np.mean(np.std(cluster_data, axis=0)))
        stds.append(np.mean(cluster_stds))

    # if fixed_k provided skip elbow detection
    if fixed_k is not None:
        return fixed_k, stds, list(k_range)

    #elbow is found if the gradient reduces enough or if there is a local minimum
    gradients = [stds[i] - stds[i+1] for i in range(len(stds)-1)]
    max_grad = max(gradients) * k_tolerance if max(gradients) > 0 else 0.01

    valid = []
    values = []

    #looping through to find k
    for i in range(1, len(gradients)):
        n_current = i + 2
        std_prev = stds[i-1]
        std = stds[i]
        try: std_next = stds[i+1]
        except: std_next = 0
        g_prev = gradients[i-1]
        g = gradients[i]

        if (n_current < min_n) or (n_current > max_n):
            continue

        #trying to satisfy conditions
        cond1 = (std <= std_next) and (std <= 0.8 * std_prev)
        cond2 = (g < max_grad) and (g_prev > g)
        if (cond1 or cond2) and std >= 0:
            valid.append(n_current)
            values.append(std)

    valid.append(int(max_n))
    if len(stds) > 0:
        values.append(stds[min(int(max_n) - 2, len(stds) - 1)] + 0.02)

    k_optimal = valid[0]
    return k_optimal, stds, list(k_range)

def find_best_clustering_method(X_cluster, methods, k_tolerance=0.15, fixed_k=None, plot_max_k=13):
    #finds the best clustering algorithm method to use depending on the standard deviation of the feature set across clusters
    max_k = 13
    method_results = {}

    #setting k range
    if cluster_by == 'iref':
        max_k = 7
        plot_max_k == 7

    #going though all clustering algorithms and finding optimum k for them, then finding the algorithm with the lowest std for their optimum k
    for method in methods:
        if method == 'spectral':
            # spectral uses CH index to find k_optimal, rather than the elbow method
            k_range = range(2, plot_max_k)  # wide range, for plotting stds/ch_scores curve

            ch_scores = []
            stds = []
            for k in k_range:
                k_labels = SpectralClustering(n_clusters=k, random_state=42).fit_predict(X_cluster)

                if len(np.unique(k_labels)) < 2:
                    ch_scores.append(np.nan)
                else:
                    ch_scores.append(calinski_harabasz_score(X_cluster, k_labels))

                cluster_stds = []
                for label in np.unique(k_labels):
                    cluster_data = X_cluster[k_labels == label]
                    cluster_stds.append(np.mean(np.std(cluster_data, axis=0)))
                stds.append(np.mean(cluster_stds))

            ch_scores = np.array(ch_scores, dtype=float)
            k_list = list(k_range)

            if fixed_k is not None:
                # force k_optimal to fixed_k, matching the other methods, for direct comparison
                k_optimal = fixed_k
            else:
                search_k_range = range(2, max_k)
                search_mask = np.array([k in search_k_range for k in k_list])
                ch_scores_restricted = np.where(search_mask, ch_scores, np.nan)
                k_optimal = k_list[np.nanargmax(ch_scores_restricted)]

        #the other algorithms use the elbow method
        else:
            k_optimal, stds, k_range = find_elbow_k_turbines(X_cluster, method, k_tolerance=k_tolerance, fixed_k=fixed_k, plot_max_k=plot_max_k)

        if method == 'kmeans':
            labels = KMeans(n_clusters=k_optimal, random_state=42, n_init=10).fit_predict(X_cluster)
        elif method == 'spectral':
            labels = SpectralClustering(n_clusters=k_optimal, random_state=42).fit_predict(X_cluster)
        elif method == 'agglomerative':
            labels = AgglomerativeClustering(n_clusters=k_optimal).fit_predict(X_cluster)
        elif method == 'gaussian':
            labels = GaussianMixture(n_components=k_optimal, random_state=42).fit_predict(X_cluster)

        # check minimum cluster size - ensure this is not exceeded
        unique, counts = np.unique(labels, return_counts=True)
        if counts.min() < min_cluster_size:
            print(f'{method}: k={k_optimal} has clusters smaller than {min_cluster_size} — trying k={k_optimal - 1}')
            k_optimal = k_optimal - 1
            if k_optimal < 2:
                print(f'{method}: could not find valid k — skipping')
                continue
            if method == 'kmeans':
                labels = KMeans(n_clusters=k_optimal, random_state=42, n_init=10).fit_predict(X_cluster)
            elif method == 'spectral':
                labels = SpectralClustering(n_clusters=k_optimal, random_state=42).fit_predict(X_cluster)
            elif method == 'agglomerative':
                labels = AgglomerativeClustering(n_clusters=k_optimal).fit_predict(X_cluster)
            elif method == 'gaussian':
                labels = GaussianMixture(n_components=k_optimal, random_state=42).fit_predict(X_cluster)

        cluster_stds = []
        for label in np.unique(labels):
            cluster_data = X_cluster[labels == label]
            cluster_stds.append(np.mean(np.std(cluster_data, axis=0)))
        mean_std = np.mean(cluster_stds)

        method_results[method] = {
            'k_optimal': k_optimal,
            'mean_std': mean_std,
            'labels': labels,
            'stds': stds,
            'k_range': k_range
        }

        print(f'{method}: k={k_optimal}, mean_std={mean_std:.4f}')

    #want the algorithm with the lowest std
    best_method = min(method_results, key=lambda x: method_results[x]['mean_std'])
    print(f'\nBest method: {best_method} with k={method_results[best_method]["k_optimal"]}')

    #returning the algorithm and results including k optimum, stds and cluster label assignments
    return best_method, method_results

def normalise(GT_df):
    #This function normalises DELs by a baseline condition, this is necessary when using the predicted DEL as feature as the models require normalised data

    #------------------------------------------------------damage normalisation---------------------------------------------------------------------
    #baseline values
    windspeed_base = 16
    iref_base = 0.14
    density_base = 1.15
    inflow_base = 0
    shear_base = 0.15

    #averaging across turbulence seeds for the damage does not work as they need to be raised to the power of the wohler exp first.
    def power_average_damage(damage_values, wohler):
        return (np.mean(damage_values ** wohler)) ** (1/wohler)


    #finding the damage value that matches the baseline environmental conditions
    #this creates a df with a corresponding baseline damage for each subset
    baseline_rows = GT_df.loc[
        (GT_df['windSpeed'] == windspeed_base) &
        (GT_df['iRef'] == iref_base) &
        (GT_df['density'] == density_base) &
        (GT_df['shearExp'] == shear_base) &
        (GT_df['inFlowAngle'] == inflow_base),
        ['node', 'load', 'wohler', 'damage']
    ]

    # apply power average across the three seeds for each subset
    #the lambda x creates another function to handle each 3 identical rows seperately, where x is each group
    baseline = baseline_rows.groupby(['node', 'load', 'wohler']).apply(
        lambda x: pd.Series({
            'baseline_damage': power_average_damage(x['damage'].values, x.name[2])
        }) ,
        include_groups=False
    ).reset_index()

    print(baseline)
    print(baseline.groupby(['node', 'load', 'wohler']).size())  # should be 1 per combination

    # merge - now one baseline value per subset, no duplicates
    GT_df = GT_df.merge(baseline, on=['node', 'load', 'wohler'], how='left')

    GT_df['damage'] = GT_df['damage'] / GT_df['baseline_damage']

    print(GT_df['damage'].describe())

    return GT_df

def centroid_deviation(GT_df, GT10_df, k_optimal, best_labels, node_features, cluster_by, env_inputs):
    #This function gets the devuation from the centroid
    #the centroid is the found from the mean env conditions in the cluster, and the DEL of the turbine closest to tehse conditions in the GT dataset is used as DEL centroid
    GT_df = GT_df.copy()
    # use only one turbulence seed
    seed = GT10_df['seedID'].unique()[0]
    GT10_df = GT10_df[GT10_df['seedID'] == seed]
    GT_df['cluster_assignment'] = best_labels

    # ---------------------------------------------------------------- COMPUTE BASELINE FOR DE-NORMALISATION ----------------------------------------------------------------
    # GT_df's predicted damage columns are normalised, so they must be de-normalised back to raw Nm before comparing against GT10's raw simulated damage values
    #baseline conditions
    windspeed_base = 16
    iref_base = 0.14
    density_base = 1.15
    inflow_base = 0
    shear_base = 0.15

    #averaging baseline DEL across turb seeds
    def power_average_damage(damage_values, wohler):
        return (np.mean(damage_values ** wohler)) ** (1 / wohler)

    baseline_rows = GT10_df.loc[
        (GT10_df['windSpeed'] == windspeed_base) &
        (GT10_df['iRef'] == iref_base) &
        (GT10_df['density'] == density_base) &
        (GT10_df['shearExp'] == shear_base) &
        (GT10_df['inFlowAngle'] == inflow_base),
        ['node', 'load', 'wohler', 'damage']
    ]

    baseline = baseline_rows.groupby(['node', 'load', 'wohler']).apply(
        lambda x: pd.Series({'baseline_damage': power_average_damage(x['damage'].values, x.name[2])}),
        include_groups=False
    ).reset_index()

   #building table for baseline damage
    baseline_lookup = {}
    for _, row in baseline.iterrows():
        col_name = f"{row['node']}_{row['load']}_{row['wohler']}"  # adjust to match your actual naming pattern
        baseline_lookup[col_name] = row['baseline_damage']

    #creating wide format from GT10 for centroid lookup - larger range to search from
    GT10_wide = GT10_df.pivot_table(
        index=env_inputs,
        columns=['node', 'load', 'wohler'],
        values='damage'
    ).reset_index()
    GT10_wide.columns = ['_'.join([str(c) for c in col]).strip('_') if col[1] != ''
                         else col[0] for col in GT10_wide.columns]

    #scaler is fitted first for finding distance
    scaler = StandardScaler()
    GT10_scaled = scaler.fit_transform(GT10_wide[env_inputs].values)

    deviation_results_pct = pd.DataFrame()
    deviation_results_abs = pd.DataFrame()

    for cluster_idx in range(k_optimal):
        cell_letter = chr(65 + cluster_idx)
        cluster_data = GT_df[GT_df['cluster_assignment'] == cluster_idx]

        centroid_env = cluster_data[env_inputs].mean().values.reshape(1, -1)
        #transform the cluster centroid using the same scaler fitted on the search pool
        centroid_env_scaled = scaler.transform(centroid_env)

        #search GT10 full dataset (standardised) for closest match to centroid
        distances = np.linalg.norm(GT10_scaled - centroid_env_scaled, axis=1)
        closest_row = GT10_wide.iloc[np.argmin(distances)]

        for subset in node_features:
            #de-normalise the predicted (normalised) damages back to raw Nm before comparing
            actual_damages = cluster_data[subset].values * baseline_lookup[subset]
            reference_damage = closest_row[subset]

            deviations_abs = np.abs(actual_damages - reference_damage)
            deviations_pct = deviations_abs / reference_damage * 100

            deviation_results_pct.loc[subset, f'{cell_letter}_mean'] = round(deviations_pct.mean(), 1)
            deviation_results_pct.loc[subset, f'{cell_letter}_max'] = round(deviations_pct.max(), 1)

            deviation_results_abs.loc[subset, f'{cell_letter}_mean'] = round(deviations_abs.mean(), 1)
            deviation_results_abs.loc[subset, f'{cell_letter}_max'] = round(deviations_abs.max(), 1)

    return deviation_results_pct, deviation_results_abs

**Defining the feature set**

In [ ]:
#defining the feature sets
if cluster_by == 'damage':
    site_damage_df = predict_damage_GT(GT_df, site_data_path, node_features)
    X_cluster = site_damage_df[node_features].values
elif cluster_by == 'iref':
    X_cluster = GT_df[['iRef']].values
elif cluster_by == 'all_env':
    scaler = StandardScaler()
    X_cluster = scaler.fit_transform(GT_df[env_inputs].values)


In [ ]:
excluded_nodes = {'stationary_hub', 'rotating_hub', 'blade_maxchord', 'tower_top'}
GT10_df = GT10_df[~GT10_df['node'].isin(excluded_nodes)]
seed = GT10_df['seedID'].unique()[0]
GT10_df = GT10_df[GT10_df['seedID'] == seed]
GT10_df_norm = normalise(GT10_df)  #normalising here outside the function in case needed


**Clustering**

In [ ]:
import matplotlib as mpl
mpl.rcParams['font.size'] = 30
mpl.rcParams['axes.titlesize'] = 30
mpl.rcParams['axes.labelsize'] = 30
mpl.rcParams['xtick.labelsize'] = 28
mpl.rcParams['ytick.labelsize'] = 28
mpl.rcParams['legend.fontsize'] = 30
mpl.rcParams['figure.dpi'] = 600

plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

#---------------------------------------------------------------------------- FINDING OPTIMUM K --------------------------------------------------------------------------------------------------------------
#optimum k occrus at the elbow of the std vs k curve:
# first running damage clustering to find optimal k
best_method, method_results = find_best_clustering_method(X_cluster, methods)


# then running all other approaches with fixed k
best_method, method_results = find_best_clustering_method(X_cluster, methods, fixed_k=k_fixed, plot_max_k=13)

k_optimal = method_results[best_method]['k_optimal']
best_labels = method_results[best_method]['labels']
k_range = method_results[best_method]['k_range']
stds = method_results[best_method]['stds']


#plotting to make sure the function finds teh right k
plt.figure(figsize=(8, 5))
plt.plot(k_range, stds, marker='o')
plt.xlabel('Number of turbine cells')
plt.ylabel('Mean intra-cluster std')
plt.title(f'Turbine clustering elbow')
plt.show()

#---------------------------------------------------------------------------- CLUSTERING --------------------------------------------------------------------------------------------------------------
# assign clusters

site_damage_df = site_damage_df.reset_index(drop=True)
site_damage_df['cluster_assignment'] = best_labels

# turbine labels are retained in site_damage_df since cluster_assignment is added as a new column
for r, row in site_damage_df.iterrows():
    cluster = int(row['cluster_assignment'])
    print(f'Turbine {row["turbine"]} assigned to cell {chr(65 + cluster)}')

# ---------------------------------------------------------------------------- FINDING DEVIATION FROM CENTROID --------------------------------------------------------------------------------------------------------

# deviation of actual damage from the centroid
if cluster_by == 'damage':
    deviation_results_pct, deviation_results_abs = centroid_deviation(
        GT_df, GT10_df, k_optimal, best_labels, all_nodes, cluster_by, env_inputs)
else:
    deviation_results_pct, deviation_results_abs = centroid_deviation(
        GT_df, GT10_df, k_optimal, best_labels, all_nodes, cluster_by, env_inputs
    )
# --------------------------------------------------------------------------------------------- SAVING RESULTS --------------------------------------------------------------------------------------------------------
GT_df['cluster_assignment'] = best_labels

if cluster_by == 'damage':
    meth = f'damage_{node_damage_features}'
else:
    meth = cluster_by

GT_df[['turbine_id', 'cluster_assignment']].to_csv(f'{site_data_path}GT_df_cluster_assignments_{meth}_{best_method}6.csv', index=False)

#all already saved
if cluster_by == 'damage' and approach == 'basic':
    deviation_results_pct.to_csv(f'{site_data_path}GT_df_deviation_{meth}_bk{k_fixed}.csv')
    deviation_results_abs.to_csv(f'{site_data_path}GT_df_deviation_{meth}_babsk{k_fixed}.csv')

elif cluster_by == 'damage' and approach == 'UNIVERSAL':
    deviation_results_pct.to_csv(f'{site_data_path}GT_df_deviation_{meth}_Uk{k_fixed}.csv')
    deviation_results_abs.to_csv(f'{site_data_path}GT_df_deviation_{meth}_Uabsk{k_fixed}.csv')

else:
     deviation_results_pct.to_csv(f'{site_data_path}GT_df_deviation_{meth}_k{k_fixed}.csv')
     deviation_results_abs.to_csv(f'{site_data_path}GT_df_deviation_{meth}_absk{k_fixed}.csv')

#------------------------------------------------------------------ PLOTTING the elbow curves for different algorithms ----------------------------------------------------------------------------------
color = "#328983"

fig, axes = plt.subplots(1, len(methods), figsize=(19, 4.8))

for i, (ax, method) in enumerate(zip(axes, methods)):
    ax.plot(method_results[method]['k_range'], method_results[method]['stds'],
            marker='o', color=color)
    ax.set_xlabel('k')
    ax.set_xticks([0, 5, 10, 15, 20, 25])
    ax.set_yticks([0.05, 0.1])
    if i == 0:
        ax.set_ylabel('Mean intra-cluster std')
    ax.text(0.98, 0.95, method.capitalize(), transform=ax.transAxes, va='top', ha='right')

plt.tight_layout()
plt.savefig(f'{figure_path}clustering_comp.pdf')
plt.show()




**Manual Damage Binning**

This section bins the turbines by damage rather than clustering. Starting from the highest damage value it bins by percentage deviation.

In [ ]:

# ------------------------------------------------------------------ DETERMINISTIC DAMAGE BINNING ------------------------------------------------------------------
bin_width_pct = 11

# user inputs
representative_subset = 'tower_base_My_4'  # subset to bin on
    
reference_damage = site_damage_df[representative_subset].values

bin_method = 'TB'

# reference is the median predicted damage across all turbines
damage_reference = reference_damage.max()

# computing percentage deviation of each turbine from reference
pct_deviation = (reference_damage - damage_reference) / damage_reference * 100

# defining bin edges based on data range and chosen bin width
# ensures the bins start on round numbers , rounds to bin edges that makes sense
min_pct = np.floor(pct_deviation.min() / bin_width_pct) * bin_width_pct
max_pct = np.ceil(pct_deviation.max() / bin_width_pct) * bin_width_pct
bin_pct = list(np.arange(min_pct, max_pct + bin_width_pct, bin_width_pct))
bin_pct[0] = -np.inf
bin_pct[-1] = np.inf

# generate labels A, B, C etc.
bin_labels = [chr(65 + i) for i in range(len(bin_pct) - 1)]

print(f'Bin edges: {bin_pct}')
print(f'Number of bins: {len(bin_labels)}')

# assign bins
GT_df['damage_bin'] = pd.cut(pct_deviation, bins=bin_pct, labels=bin_labels)
print(GT_df.groupby('damage_bin').size())

# compute intra-bin deviation using actual damage
bin_deviation_results_pct = pd.DataFrame()
bin_deviation_results_abs = pd.DataFrame()

for bin_label in bin_labels:
    bin_data = GT_df[GT_df['damage_bin'] == bin_label]
    if len(bin_data) == 0:
        continue

    # finds representative turbine closest to bin centre
    # finds the percentage deviation in the middle of the bin, converts this back to damage then finds the turbine closest to this value
    bin_idx = bin_labels.index(bin_label)
    bin_centre_pct = (bin_pct[bin_idx] + bin_pct[bin_idx + 1]) / 2
    if not np.isfinite(bin_centre_pct):
        bin_centre_pct = bin_pct[bin_idx + 1] - bin_width_pct / 2 if not np.isfinite(bin_pct[bin_idx]) else bin_pct[bin_idx] + bin_width_pct / 2
    bin_centre_damage = damage_reference * (1 + bin_centre_pct / 100)

    closest_idx = np.argmin(np.abs(bin_data[representative_subset].values - bin_centre_damage))
    representative_turbine = bin_data.iloc[closest_idx]

    for subset in all_nodes:
        if subset not in bin_data.columns:
            continue
        actual_damages = bin_data[subset].values
        reference = representative_turbine[subset]
        deviations_abs = np.abs(actual_damages - reference)
        deviations_pct = deviations_abs / reference * 100

        bin_deviation_results_pct.loc[subset, f'{bin_label}_mean'] = round(deviations_pct.mean(), 1)
        bin_deviation_results_pct.loc[subset, f'{bin_label}_max'] = round(deviations_pct.max(), 1)

        bin_deviation_results_abs.loc[subset, f'{bin_label}_mean'] = round(deviations_abs.mean(), 1)
        bin_deviation_results_abs.loc[subset, f'{bin_label}_max'] = round(deviations_abs.max(), 1)

if approach == 'UNIVERSAL':
    a = 'U'
else:
    a = 'b'

bin_deviation_results_pct.to_csv(f'{site_data_path}damage_binning_deviation_{bin_width_pct}_{bin_method}_{a}_pct.csv')
bin_deviation_results_abs.to_csv(f'{site_data_path}damage_binning_deviation_{bin_width_pct}_{bin_method}_{a}_abs.csv')


In [ ]:
#These are the names to find the right files for each feature set for plotting

all_env_name = f'all_env_'
iref_name = f'iref_'
tB_name_b = f'damage_towerBase_b'
all_name_b = f'damage_all_b'
mxmy_name_b = f'damage_MxMy_b'
tB_name_u = f'damage_towerBase_U'
all_name_u = f'damage_all_U'
mxmy_name_u = f'damage_MxMy_U'

**plotting basic vs universal model damage feature set results**

In [ ]:
#setting the appearace of the plots
mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['legend.fontsize'] = 12
mpl.rcParams['figure.dpi'] = 600

plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'


pfe_color = "#2C706B"  # Basic
col = "#2AB9AD"        # Universal

k = 7

subsets = ['blade_root_Mx_14', 'blade_root_My_14', 'tower_base_My_4']
group_labels = ['Blade root\nMx 14', 'Blade root\nMy 14', 'Tower base\nMy 4']

def plot_u_vs_b_bar(feature_set_name, u_path, b_path, save_path):
    df_u = pd.read_csv(u_path, index_col=0)
    df_b = pd.read_csv(b_path, index_col=0)

    mean_cols_u = [c for c in df_u.columns if 'mean' in c]
    mean_cols_b = [c for c in df_b.columns if 'mean' in c]

    vals_u = [df_u.loc[s, mean_cols_u].values.astype(float).mean() for s in subsets]
    vals_b = [df_b.loc[s, mean_cols_b].values.astype(float).mean() for s in subsets]

    print(f'\n{feature_set_name}')
    for s, vu, vb in zip(subsets, vals_u, vals_b):
        print(f'  {s}: Universal={vu:.2f}%, Basic={vb:.2f}%, difference (U-B)={vu - vb:.2f}%')

    x = np.arange(len(subsets))
    width = 0.2

    fig, ax = plt.subplots(figsize=(5.2, 2.9))
    ax.bar(x - width/2, vals_u, width, label='Universal', color=col)
    ax.bar(x + width/2, vals_b, width, label='Basic', color=pfe_color)

    ax.set_xticks(x)
    ax.set_xticklabels(group_labels)
    ax.set_ylabel('Mean % intra-cluster DEL variation')
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()


# --- call once per feature set ---
plot_u_vs_b_bar('All ten subsets', f'{site_data_path}GT_df_deviation_{all_name_u}k{k}.csv',
                 f'{site_data_path}GT_df_deviation_{all_name_b}k{k}.csv',
                 f'{figure_path}u_vs_b_bar_all_mean5.pdf')

plot_u_vs_b_bar('Three subsets', f'{site_data_path}GT_df_deviation_{mxmy_name_u}k{k}.csv',
                 f'{site_data_path}GT_df_deviation_{mxmy_name_b}k{k}.csv',
                 f'{figure_path}u_vs_b_bar_mxmy_mean5.pdf')

plot_u_vs_b_bar('Single subset', f'{site_data_path}GT_df_deviation_{tB_name_u}k{k}.csv',
                 f'{site_data_path}GT_df_deviation_{tB_name_b}k{k}.csv',
                 f'{figure_path}u_vs_b_bar_tB_mean5.pdf')

**plotting DSC vs environmental clustering results**
for this to run properly all feature sets have to have been run in the above clustering code i.e. damage (all, MxMy and towerBase) and iref and all env

In [ ]:
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.titlesize'] = 10
mpl.rcParams['axes.labelsize'] = 10
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 10
mpl.rcParams['figure.dpi'] = 600
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

col_dsc10 = "#28A99E"
col_dsc1 = "#8834C9C6"
col_dsc3 = "#1864CE"
col_iref = "#C0320AC5"
col_env = "#F0A22E"
col_bin = "#6BB052"

approaches = [all_name_u, mxmy_name_u, tB_name_u, 'bin_u', all_env_name, iref_name,]
labels = ['DSC \n (10 subsets)', 'DSC \n (3 subsets)', 'DSC \n (1 subset)', 'DEL binning', 'All env\nvariables', 'Turbulence\nintensity',]
approach_colors = [col_dsc10, col_dsc3, col_dsc1, col_bin, col_env, col_iref,]

mean_devs = {}
max_devs = {}

bin_method = 'TB'
k = 7

for approach in approaches:
    if approach == 'bin_u':
        df = pd.read_csv(f'{site_data_path}damage_binning_deviation_11_average_U_Pct.csv', index_col=0)
    else:
        df = pd.read_csv(f'{site_data_path}GT_df_deviation_{approach}k{k}.csv', index_col=0)

    mean_cols = [c for c in df.columns if 'mean' in c]
    max_cols = [c for c in df.columns if 'max' in c]

    mean_devs[approach] = df[mean_cols].values.mean()
    max_devs[approach] = df[max_cols].values.mean()

    print(f'{approach}: mean={mean_devs[approach]:.2f}%, max={max_devs[approach]:.2f}%')

x = np.arange(len(approaches))
width = 0.3

fig, ax = plt.subplots(figsize=(6.27, 3.5))
ax.bar(x - width/2, [mean_devs[a] for a in approaches], width,
       color=approach_colors, alpha=0.7, linewidth=0.4)
ax.bar(x + width/2, [max_devs[a] for a in approaches], width,
       color=approach_colors, alpha=0.7, edgecolor="#797A79", linewidth=0.4, hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('% Intra-cluster DEL variation')

# custom legend showing mean/max pattern distinction only (not per-approach color)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='white', edgecolor="#797A79", linewidth=0.4, label='Mean variation'),
    Patch(facecolor='white', edgecolor="#797A79", linewidth=0.4, hatch='//', label='Max variation'),
]
ax.legend(handles=legend_elements, frameon=False)

plt.tight_layout()
#plt.savefig(f'C:/Users/Saoirse/OneDrive - University of Strathclyde/Documents/COLLEGE/MASTERS/THESIS/FIGURES/feature_scenario_comparison5.pdf')
plt.show()

**plotting uncertainty vs k**

In [ ]:
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.titlesize'] = 10
mpl.rcParams['axes.labelsize'] = 10
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 9
mpl.rcParams['figure.dpi'] = 600
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

col_dsc10 = "#2AB9AD"
col_dsc1 = "#8544B8C5"
col_dsc3 = "#2596CB"
col_env = "#F0A22E"
col_iref = "#C0320AC5"


approaches = [all_name_u, mxmy_name_u, tB_name_u, iref_name, all_env_name]
approach_labels = ['DSC (10 subsets)', 'DSC (3 subsets)', 'DSC (1 subset)',
                    'Turbulence intensity', 'All environmental variables']
approach_colors = [col_dsc10, col_dsc3, col_dsc1, col_iref, col_env]

approach_labels = ['DSC (10 ft.)', 'DSC (3 ft.)', 'DSC (1 ft.)',
                    'Turbulence intensity', 'All env. vars.']

chosen_k = {
    all_name_u: 6,
    mxmy_name_u: 7,
    tB_name_u: 7,
    iref_name: 7,
    all_env_name: 10,
}

full_k_range = range(2, 13)

fig, ax = plt.subplots(figsize=(6.27, 3.5))

for approach, label, color in zip(approaches, approach_labels, approach_colors):
    k_values = range(2, 8) if approach == iref_name else range(2, 13)

    mean_devs_k = []
    valid_k = []

    for k in k_values:
        df = pd.read_csv(f'{site_data_path}GT_df_deviation_{approach}k{k}.csv', index_col=0)
        mean_cols = [c for c in df.columns if 'mean' in c]
        mean_devs_k.append(df[mean_cols].values.mean())
        valid_k.append(k)

    line, = ax.plot(valid_k, mean_devs_k, marker='o', markersize=3, label=label, color=color)

    if approach in chosen_k:
        k_star = chosen_k[approach]
        if k_star in valid_k:
            idx = valid_k.index(k_star)
            #ax.plot(valid_k[idx], mean_devs_k[idx], marker='*', markersize=16,color=line.get_color(), markeredgecolor='none')

ax.set_xlabel('k')
ax.set_ylabel('% Mean intra-cluster DEL variation')
ax.set_xticks(list(full_k_range))
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f'{figure_path}deviation_vs_k_all_approaches.pdf')
plt.show()

**Plotting the intra-cluster variance for the different DSC feature subsets**

In [ ]:
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.titlesize'] = 10
mpl.rcParams['axes.labelsize'] = 10
mpl.rcParams['xtick.labelsize'] = 9
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 10
mpl.rcParams['figure.dpi'] = 600
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['mathtext.fontset'] = 'stix'

col_dsc10 = "#27ADA2"
col_dsc1 = "#7F31BB"
col_dsc3 = "#266DCF"

dsc_approaches = [all_name_u, mxmy_name_u, tB_name_u]
dsc_labels = ['DSC (10 subsets)', 'DSC (3 subsets)', 'DSC (1 subset)']
dsc_colors = [col_dsc10, col_dsc3, col_dsc1]

k = 7

# all ten node/load/Wohler subsets
subsets = ['blade_root_Mx_4', 'blade_root_Mx_9', 'blade_root_Mx_10', 'blade_root_Mx_14',
           'blade_root_My_4', 'blade_root_My_9', 'blade_root_My_10', 'blade_root_My_14',
           'tower_base_My_4', 'tower_base_My_9']
subset_labels = ['BR \n Mx 4', 'BR \n Mx 9', 'BR \n Mx 10', 'BR \n Mx 14', 'BR \n My 4', 'BR \n My 9', 'BR \n My 10', 'BR \n My 14', 'TB \n My 4', 'TB \n My 9']

# load data for each DSC approach
approach_data = {}
for approach in dsc_approaches:
    df = pd.read_csv(f'{site_data_path}GT_df_deviation_{approach}k{k}.csv', index_col=0)
    mean_cols = [c for c in df.columns if 'mean' in c]
    approach_data[approach] = {s: df.loc[s, mean_cols].values.astype(float).mean() for s in subsets}

x = np.arange(len(subsets))
width = 0.25

fig, ax = plt.subplots(figsize=(6.27, 3.5))

for i, (approach, label, color) in enumerate(zip(dsc_approaches, dsc_labels, dsc_colors)):
    vals = [approach_data[approach][s] for s in subsets]
    offset = (i - 1) * width
    ax.bar(x + offset, vals, width, label=label, color=color, alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(subset_labels)
ax.set_ylabel('% Mean intra-cluster DEL variation')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f'{figure_path}dsc_subset_comparison.pdf')
plt.show()